# Debugging code
Week 4 - Discussion section

In [1]:
import io
import pandas as pd

CSV = """site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
SB-01,2024-01-15,12.4,"1,240",0.82,good
SB-01,2024-02-15,13.1,"1,180",0.79,good
SB-01,2024-03-15,15.8,"2,050",1.21,good
SB-01,2024-04-15,18.2,"1,620",0.94,good
sb-01,2024-05-15,21.5,"980",0.66,good
SB-01 ,2024-06-15,-999,"540",0.51,suspect
SB-02,2024-01-15,11.8,"3,410",1.88,good
SB-02,2024-02-15,12.3,"3,290",1.92,good
SB-02,2024-03-15,14.9,"4,870",2.41,good
SB-02,2024-04-15,17.6,"3,950",2.09,estimated
SB-02,2024-05-15,20.9,"2,240",1.44,good
sb-02,2024-06-15,-999,"1,130",1.02,suspect
SB-03,2024-01-15,10.2,"620",<0.05,good
SB-03,2024-02-15,10.9,"590",<0.05,good
SB-03,2024-03-15,13.4,"1,040",0.33,good
SB-03,2024-04-15,16.1,"870",0.28,good
SB-03 ,2024-05-15,19.3,"410",<0.05,good
SB-03,2024-06-15,-999,"180",<0.05,suspect
"""

stream = pd.read_csv(io.StringIO(CSV))

In [ ]:
stream

How many monitoring sites are in this dataset? 3

What is a plausible range for stream water temperature in degrees Celsius? 10-22

Roughly what mean water temperature do you expect across all measurements? 15

In [ ]:
# Convert stream discharge from cubic feet per second to cubic metres per second
stream['discharge_cfs'] * 0.0283168

In [ ]:
stream.dtypes # discharge_cfs is object

In [ ]:
# Isolate: Reduce to the smallest failing piece, which here is a single value
stream['discharge_cfs'].iloc[0] * 0.0283168  # same TypeError, one value
stream['discharge_cfs'].iloc[0]              # '1,240'

In [ ]:
# Number of unique values in the discharge column
stream['discharge_cfs'].unique()

In [ ]:
# Fix and verify: Remove the separator, convert, then check the result
discharge_cms = (stream['discharge_cfs']
                    .str.replace(',', '', regex=False) # Replace commas with nothing. regex=False
                    .astype(float)) * 0.0283168 # Change the type to float and convert

discharge_cms.head(3)

In [ ]:
# Add a guard. assert that this is a numeric column. do this before run code. 
assert pd.api.types.is_numeric_dtype(discharge_cms), 'Discharge failed to convert to numeric'

In [ ]:
# Building a minimal reproducible example, which is a two-row version of the problem that another person can run in isolation
import pandas as pd
df = pd.DataFrame({'discharge_cfs': ['1,240', '980']})
df['discharge_cfs'] * 0.0283168     # TypeError: can't multiply sequence by non-int

In [ ]:
# Attempt to convert nitrate column to a float datatype
stream['nitrate_mg_l'].astype(float)

In [ ]:
# Look at the datatype. It is a string
print(stream['nitrate_mg_l'].dtype)

In [ ]:
# Look at the values of the column to see what is wrong
stream['nitrate_mg_l'] # Contains '<', making it a string, so have to remove those

In [ ]:
# Fix the issue by setting those values to 0.05
stream['nitrate_mg_l'] = (stream['nitrate_mg_l']
                        .str.replace('<', '', regex=False) # Replace < with nothing. RegEx can be used to check if a string contains the specified search pattern
                        .astype(float)) # Change the type to float and convert

In [ ]:
# Re-check the datatype
print(stream['nitrate_mg_l'].dtype)

In [ ]:
stream['water_temp_c'].mean() # Get a negative value because of -999

In [ ]:
import numpy as np
stream['water_temp_c'].replace(-999.0, np.nan)

In [2]:
stream['site_id'].nunique() # Doesn't match our expectaion of 3 sites

7

In [3]:
stream['site_id'] = stream['site_id'].str.strip().str.lower() # Remove the spaces and convert to lowercase

In [4]:
stream['site_id'].nunique() # Re-check and see that there are now 3 sites

3

In [6]:
stream['site_id'].unique()

array(['sb-01', 'sb-02', 'sb-03'], dtype=object)

In [ ]:
site_means = stream.groupby('site_id')['water_temp_c'].mean()
stream['site_mean'] = site_means
stream[['site_id', 'water_temp_c', 'site_mean']].head()

In [ ]:
site_means